# 🧭 Chapter 07-01a: LangGraph Fundamentals — State, Nodes & Edges

---

## 📖 The Story: Building a City's Traffic System

Imagine you're designing a city's traffic system. Each **intersection** is a *node* (it does something — directs traffic). Each **road** is an *edge* (connects two intersections). The **traffic signals** are *conditional edges* (go left if heavy traffic, go straight otherwise).

And the **state** of the entire system? That's the snapshot of every car, every signal, every road at any moment.

That's exactly how LangGraph works — but for AI agents.

---

## 🎯 What is LangGraph?

```
╔══════════════════════════════════════════════════════════════════════╗
║  LANGGRAPH                                                          ║
║                                                                      ║
║  A framework for building stateful, multi-step agent workflows     ║
║  as GRAPHS. Unlike chains (linear), graphs support cycles,         ║
║  conditionals, and complex control flow.                            ║
║                                                                      ║
║  Chain: A → B → C (done)                                           ║
║  Graph: A → B → C → maybe back to A (loops, decisions!)           ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 🧠 The Three Pillars

### 1. State (TypedDict)
A shared dictionary that flows through every node:
```python
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]  # Append, don't replace
    user_intent: str
```

**`Annotated[list, add_messages]`** = the reducer. When a node returns `{"messages": [new_msg]}`, it APPENDS to the list instead of replacing it.

### 2. Nodes (Functions)
Each node is a function that takes State, does something, returns partial State:
```python
def greet(state: State) -> dict:
    return {"messages": [AIMessage(content="Hello!")]}
```

### 3. Edges (Connections)
- **Normal edge**: `graph.add_edge("A", "B")` — always go from A to B
- **Conditional edge**: `graph.add_conditional_edges("A", router_fn)` — go to B or C based on state

---

## 🔄 Why Not Just LCEL Chains?

| Feature | LCEL Chains | LangGraph |
|---------|-------------|------------|
| Flow | Linear (A→B→C) | Graphs (cycles, branches) |
| Loops | ❌ Not possible | ✅ Cycles built-in |
| State | Passed through pipe | Shared TypedDict |
| Persistence | ❌ | ✅ Checkpoints |
| Human-in-loop | Hard | ✅ First-class support |
| Debugging | Hard to inspect | ✅ Step-by-step visibility |

**Rule of thumb**: Simple pipeline → LCEL. Agent with decisions/loops → LangGraph.

---

## 🏗️ Graph Building Pattern

```
1. Define State (TypedDict)
2. Create StateGraph(State)
3. Add nodes: graph.add_node("name", function)
4. Add edges: graph.add_edge("from", "to")
5. Set entry: graph.set_entry_point("first_node")
6. Set finish: graph.set_finish_point("last_node")  (or use END)
7. Compile: app = graph.compile()
8. Run: app.invoke({"messages": [...]})
```

```
    ┌──────────┐
    │  START    │
    └────┬─────┘
         │
    ┌────▼─────┐
    │  Agent   │◄──────┐
    └────┬─────┘       │
         │             │
    ┌────▼─────┐  ┌────┴─────┐
    │ Should   │  │  Tools   │
    │ continue?│──►          │
    └────┬─────┘  └──────────┘
         │ No
    ┌────▼─────┐
    │   END    │
    └──────────┘
```

---

## 🎓 Interview Corner

**Q: When would you use LangGraph over a simple LCEL chain?**

<details>
<summary>💡 Click for Expert Answer</summary>

Use LangGraph when you need:
1. **Cycles/loops** — e.g., agent retries, iterative refinement
2. **Conditional branching** — different paths based on LLM output
3. **Persistence** — save and resume workflows (checkpointing)
4. **Human-in-the-loop** — pause for human approval before executing
5. **Multi-agent coordination** — multiple agents with handoffs

LCEL is sufficient for linear pipelines: prompt → LLM → parser.

</details>

---

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 LangGraph = stateful agent workflows as graphs.                │
├─────────────────────────────────────────────────────────────────────┤
│  📌 State (TypedDict) + Nodes (functions) + Edges (connections).   │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Annotated[list, add_messages] = append reducer for messages.   │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Conditional edges enable branching based on state.             │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Use LCEL for pipelines, LangGraph for agents with loops.       │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Hands-On — Build a Chatbot with Routing!

---

### 🎉 You Get Graphs!
From linear chains to cyclic graphs — this is how real agents work. 🚀